# Arsenal-spillere i FPL 2025/26

Denne notebooken viser spilleroversikten, sesongstatistikk, xG/xA og kamp-for-kamp-data for Arsenal. Kjør cellene ovenfra og ned.

In [1]:
%pip install -q pandas


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

candidates = [Path.cwd() / "data-source" / "data", Path.cwd().parent / "data-source" / "data"]
DATA_DIR = next((path for path in candidates if path.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Fant ikke data-source/data")

SEASON = "2025-26"
season_dir = DATA_DIR / SEASON
print("Datamappe:", season_dir)

Datamappe: /Users/henrik/Documents/fplmodell/data-source/data/2025-26


## Last inn tabellene

In [10]:
players = pd.read_csv(season_dir / "players_raw.csv")
teams = pd.read_csv(season_dir / "teams.csv")
gameweeks = pd.read_csv(season_dir / "gws" / "merged_gw.csv")
fixtures = pd.read_csv(season_dir / "fixtures.csv")

arsenal_team = teams.loc[teams["name"].eq("Arsenal")].iloc[0]
arsenal_id = arsenal_team["id"]
arsenal_players = players.loc[players["team"].eq(arsenal_id)].copy()
arsenal_gw = gameweeks.loc[gameweeks["team"].eq("Arsenal")].copy()

print(f"Arsenal har lag-ID {arsenal_id}")
print(f"{len(arsenal_players)} spillere i sesongoversikten")
print(f"{len(arsenal_gw):,} spiller-kamp-rader i gameweek-dataene")

Arsenal har lag-ID 1
36 spillere i sesongoversikten
1,352 spiller-kamp-rader i gameweek-dataene


## Arsenal-troppen
Prisfeltene i FPL-data er lagret i tideler av en million.

In [11]:
position_names = {1: "GK", 2: "DEF", 3: "MID", 4: "FWD"}
arsenal_players["name"] = (arsenal_players["first_name"] + " " + arsenal_players["second_name"]).str.strip()
arsenal_players["position"] = arsenal_players["element_type"].map(position_names)
arsenal_players["price_m"] = arsenal_players["now_cost"] / 10

squad_columns = [
    "id", "name", "web_name", "position", "price_m", "status",
    "minutes", "starts", "total_points", "points_per_game",
    "goals_scored", "assists", "clean_sheets", "bonus",
    "expected_goals", "expected_assists",
    "expected_goal_involvements", "selected_by_percent"
]
display(
    arsenal_players[squad_columns]
    .sort_values(["position", "total_points"], ascending=[True, False])
    .reset_index(drop=True)
)

,id,name,web_name,position,price_m,status,minutes,starts,total_points,points_per_game,goals_scored,assists,clean_sheets,bonus,expected_goals,expected_assists,expected_goal_involvements,selected_by_percent
0,5,Gabriel dos Santos Magalhães,Gabriel,DEF,7.3,a,2750,30,209,6.5,3,5,18,30,2.94,1.75,4.69,45.4
1,8,Jurriën Timber,J.Timber,DEF,6.0,d,2452,28,149,5.0,3,6,13,9,4.71,1.53,6.24,15.2
2,6,William Saliba,Saliba,DEF,6.3,a,2614,30,137,4.4,1,0,15,12,0.88,1.21,2.09,17.1
3,7,Riccardo Calafiori,Calafiori,DEF,5.6,a,1697,22,109,4.2,1,2,13,6,3.36,0.76,4.12,6.0
4,725,Piero Hincapié,Hincapie,DEF,5.1,a,1787,20,87,3.5,1,2,5,7,0.36,1.67,2.03,0.9
5,11,Benjamin White,White,DEF,5.1,i,699,9,45,3.8,0,1,5,3,0.45,0.69,1.14,0.5
6,662,Cristhian Mosquera,Mosquera,DEF,5.3,a,986,9,40,2.0,0,0,3,1,0.10,0.63,0.73,0.3
7,10,Myles Lewis-Skelly,Lewis-Skelly,DEF,5.0,a,697,5,29,1.4,0,0,2,0,0.10,0.20,0.30,1.9
8,9,Jakub Kiwior,Kiwior,DEF,5.4,u,0,0,0,0.0,0,0,0,0,0.00,0.00,0.00,0.0
9,13,Brayden Clarke,Clarke,DEF,3.7,a,0,0,0,0.0,0,0,0,0,0.00,0.00,0.00,0.5


## Samlet kampstatistikk per spiller
Denne tabellen summerer alle kamp-radene og inkluderer spillere som representerte Arsenal i løpet av sesongen.

In [12]:
numeric_columns = [
    "minutes", "starts", "total_points", "goals_scored", "assists",
    "clean_sheets", "bonus", "bps", "expected_goals",
    "expected_assists", "expected_goal_involvements",
    "expected_goals_conceded"
]
for column in numeric_columns:
    arsenal_gw[column] = pd.to_numeric(arsenal_gw[column], errors="coerce").fillna(0)

summary = (
    arsenal_gw.groupby(["element", "name", "position"], as_index=False)[numeric_columns]
    .sum()
)
summary["points_per_90"] = summary["total_points"] / summary["minutes"].replace(0, pd.NA) * 90
summary["xg_per_90"] = summary["expected_goals"] / summary["minutes"].replace(0, pd.NA) * 90
summary["xa_per_90"] = summary["expected_assists"] / summary["minutes"].replace(0, pd.NA) * 90
summary["xgi_per_90"] = summary["expected_goal_involvements"] / summary["minutes"].replace(0, pd.NA) * 90

display(
    summary.sort_values("total_points", ascending=False)
    .round(2)
    .reset_index(drop=True)
)

,element,name,position,minutes,starts,total_points,goals_scored,assists,clean_sheets,bonus,bps,expected_goals,expected_assists,expected_goal_involvements,expected_goals_conceded,points_per_90,xg_per_90,xa_per_90,xgi_per_90
0,5,Gabriel dos Santos Magalhães,DEF,2750,30,209,3,5,18,30,724,2.94,1.75,4.69,22.01,6.84,0.096218,0.057273,0.153491
1,21,Declan Rice,MID,3093,35,184,4,9,18,23,790,3.15,7.32,10.47,23.87,5.354025,0.091659,0.212997,0.304656
2,1,David Raya Martín,GK,3330,37,162,0,0,19,11,633,0.00,0.07,0.07,27.56,4.378378,0.0,0.001892,0.001892
3,16,Bukayo Saka,MID,2218,25,157,7,10,12,18,570,7.57,7.16,14.73,15.57,6.370604,0.307169,0.290532,0.597701
4,8,Jurriën Timber,DEF,2452,28,149,3,6,13,9,532,4.71,1.53,6.24,17.45,5.469005,0.172879,0.056158,0.229038
5,6,William Saliba,DEF,2614,30,137,1,0,15,12,581,0.88,1.21,2.09,20.41,4.716909,0.030298,0.04166,0.071959
6,26,Martín Zubimendi Ibáñez,MID,2991,34,133,5,1,16,9,604,2.84,2.27,5.11,26.09,4.002006,0.085456,0.068305,0.153761
7,666,Viktor Gyökeres,FWD,2217,26,128,14,1,12,16,474,12.26,1.94,14.19,17.81,5.196211,0.4977,0.078755,0.576049
8,20,Leandro Trossard,MID,1997,21,119,6,6,11,11,445,5.60,3.59,9.18,13.94,5.363045,0.252379,0.161793,0.413721
9,266,Eberechi Eze,MID,1802,21,110,7,3,10,8,429,4.94,2.66,7.60,12.99,5.493896,0.246726,0.132852,0.379578


## Ledende Arsenal-spillere

In [13]:
for metric, title in {
    "total_points": "Flest FPL-poeng",
    "expected_goals": "Høyest xG",
    "expected_assists": "Høyest xA",
    "expected_goal_involvements": "Høyest xGI",
    "points_per_90": "Flest poeng per 90",
}.items():
    print(f"\n{title}")
    display(
        summary.loc[summary["minutes"] >= 180, ["name", "position", "minutes", metric]]
        .sort_values(metric, ascending=False)
        .head(10)
        .round(2)
        .reset_index(drop=True)
    )


Flest FPL-poeng


,name,position,minutes,total_points
0,Gabriel dos Santos Magalhães,DEF,2750,209
1,Declan Rice,MID,3093,184
2,David Raya Martín,GK,3330,162
3,Bukayo Saka,MID,2218,157
4,Jurriën Timber,DEF,2452,149
5,William Saliba,DEF,2614,137
6,Martín Zubimendi Ibáñez,MID,2991,133
7,Viktor Gyökeres,FWD,2217,128
8,Leandro Trossard,MID,1997,119
9,Eberechi Eze,MID,1802,110



Høyest xG


,name,position,minutes,expected_goals
0,Viktor Gyökeres,FWD,2217,12.26
1,Bukayo Saka,MID,2218,7.57
2,Leandro Trossard,MID,1997,5.60
3,Eberechi Eze,MID,1802,4.94
4,Jurriën Timber,DEF,2452,4.71
5,Gabriel Martinelli Silva,MID,1065,4.12
6,Riccardo Calafiori,DEF,1697,3.36
7,Mikel Merino Zazón,MID,1024,3.36
8,Kai Havertz,FWD,577,3.31
9,Declan Rice,MID,3093,3.15



Høyest xA


,name,position,minutes,expected_assists
0,Declan Rice,MID,3093,7.32
1,Bukayo Saka,MID,2218,7.16
2,Martin Ødegaard,MID,1363,3.65
3,Leandro Trossard,MID,1997,3.59
4,Noni Madueke,MID,1205,2.86
5,Eberechi Eze,MID,1802,2.66
6,Martín Zubimendi Ibáñez,MID,2991,2.27
7,Viktor Gyökeres,FWD,2217,1.94
8,Gabriel dos Santos Magalhães,DEF,2750,1.75
9,Piero Hincapié,DEF,1787,1.67



Høyest xGI


,name,position,minutes,expected_goal_involvements
0,Bukayo Saka,MID,2218,14.73
1,Viktor Gyökeres,FWD,2217,14.19
2,Declan Rice,MID,3093,10.47
3,Leandro Trossard,MID,1997,9.18
4,Eberechi Eze,MID,1802,7.60
5,Jurriën Timber,DEF,2452,6.24
6,Gabriel Martinelli Silva,MID,1065,5.39
7,Martín Zubimendi Ibáñez,MID,2991,5.11
8,Martin Ødegaard,MID,1363,4.90
9,Mikel Merino Zazón,MID,1024,4.81



Flest poeng per 90


,name,position,minutes,points_per_90
0,Gabriel dos Santos Magalhães,DEF,2750,6.84
1,Bukayo Saka,MID,2218,6.370604
2,Mikel Merino Zazón,MID,1024,5.888672
3,Benjamin White,DEF,699,5.793991
4,Riccardo Calafiori,DEF,1697,5.78079
5,Kai Havertz,FWD,577,5.615251
6,Eberechi Eze,MID,1802,5.493896
7,Jurriën Timber,DEF,2452,5.469005
8,Noni Madueke,MID,1205,5.377593
9,Leandro Trossard,MID,1997,5.363045


## Kamp-for-kamp-data

In [14]:
match_columns = [
    "GW", "kickoff_time", "name", "position", "opponent_team",
    "was_home", "minutes", "starts", "goals_scored", "assists",
    "expected_goals", "expected_assists", "total_points", "value"
]
display(
    arsenal_gw[match_columns]
    .sort_values(["GW", "kickoff_time", "name"])
    .reset_index(drop=True)
)

,GW,kickoff_time,name,position,opponent_team,was_home,minutes,starts,goals_scored,assists,expected_goals,expected_assists,total_points,value
0,1,2025-08-17T15:30:00Z,Albert Sambi Lokonga,MID,14,False,0,0,0,0,0.00,0.00,0,45
1,1,2025-08-17T15:30:00Z,Benjamin White,DEF,14,False,70,1,0,0,0.01,0.01,6,55
2,1,2025-08-17T15:30:00Z,Brayden Clarke,DEF,14,False,0,0,0,0,0.00,0.00,0,40
3,1,2025-08-17T15:30:00Z,Bukayo Saka,MID,14,False,90,1,0,0,0.15,0.09,3,100
4,1,2025-08-17T15:30:00Z,Christian Nørgaard,MID,14,False,0,0,0,0,0.00,0.00,0,55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1347,38,2026-05-24T15:00:00Z,Piero Hincapié,DEF,8,False,90,1,0,0,0.03,0.02,2,51
1348,38,2026-05-24T15:00:00Z,Riccardo Calafiori,DEF,8,False,45,1,0,0,0.00,0.10,1,56
1349,38,2026-05-24T15:00:00Z,Tommy Setford,GK,8,False,0,0,0,0,0.00,0.00,0,39
1350,38,2026-05-24T15:00:00Z,Viktor Gyökeres,FWD,8,False,7,0,0,0,0.16,0.10,1,91


## Undersøk én spiller

In [15]:
PLAYER_NAME = "Saka"  # Endre navnet her

player_matches = arsenal_gw.loc[
    arsenal_gw["name"].str.contains(PLAYER_NAME, case=False, na=False),
    match_columns,
].sort_values(["GW", "kickoff_time"])

print(f"Fant {len(player_matches)} kamper for søket '{PLAYER_NAME}'")
display(player_matches.reset_index(drop=True))

Fant 38 kamper for søket 'Saka'


,GW,kickoff_time,name,position,opponent_team,was_home,minutes,starts,goals_scored,assists,expected_goals,expected_assists,total_points,value
0,1,2025-08-17T15:30:00Z,Bukayo Saka,MID,14,False,90,1,0,0,0.15,0.09,3,100
1,2,2025-08-23T16:30:00Z,Bukayo Saka,MID,11,True,52,1,1,0,0.11,0.03,6,100
2,3,2025-08-31T15:30:00Z,Bukayo Saka,MID,12,False,0,0,0,0,0.00,0.00,0,99
3,4,2025-09-13T11:30:00Z,Bukayo Saka,MID,16,True,0,0,0,0,0.00,0.00,0,98
4,5,2025-09-21T15:30:00Z,Bukayo Saka,MID,13,True,45,0,0,0,0.04,0.21,1,98
5,6,2025-09-28T15:30:00Z,Bukayo Saka,MID,15,False,69,1,0,0,0.08,0.01,2,98
6,7,2025-10-04T14:00:00Z,Bukayo Saka,MID,19,True,90,1,1,0,0.84,1.11,8,98
7,8,2025-10-18T16:30:00Z,Bukayo Saka,MID,10,False,90,1,0,0,0.07,0.55,7,99
8,9,2025-10-26T14:00:00Z,Bukayo Saka,MID,8,True,65,1,0,0,0.02,0.01,3,100
9,10,2025-11-01T15:00:00Z,Bukayo Saka,MID,3,False,90,1,0,0,0.58,0.06,3,101
